In [20]:
import matplotlib.pyplot as plt
import torch
import numpy
import torch.nn.functional as F
%matplotlib inline

In [21]:
words = open('names.txt', 'r').read().splitlines()

In [22]:
chars = ['.'] + sorted(list(set(''.join(words))))
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s,i in stoi.items()}

In [23]:
# training split, dev/validation split, test split
# 80%, 10%, 10%

# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # crop and append
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [24]:
embedding_dimensions = 20
hidden_layer_size = 200

g = torch.Generator().manual_seed(2147283647)
C = torch.randn(27, embedding_dimensions, generator=g)
W1 = torch.randn((embedding_dimensions * block_size, hidden_layer_size), generator=g)
b1 = torch.randn(hidden_layer_size, generator=g)
W2 = torch.randn((hidden_layer_size, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [25]:
sum(p.nelement() for p in parameters) # number of parameters in total

18167

In [26]:
for p in parameters:
    p.requires_grad = True

In [27]:
for i in range(10_000):
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (32,))

    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, embedding_dimensions * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    print(i, loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.01 * p.grad

0 24.500991821289062
1 31.02631187438965
2 29.72682762145996
3 31.768539428710938
4 22.515512466430664
5 24.38011360168457
6 26.010770797729492
7 28.231876373291016
8 28.083768844604492
9 26.495750427246094
10 25.30092430114746
11 26.33820915222168
12 24.300973892211914
13 27.399707794189453
14 28.874784469604492
15 23.85553550720215
16 26.0810604095459
17 24.252426147460938
18 21.358083724975586
19 26.740001678466797
20 28.86297035217285
21 21.352458953857422
22 22.022022247314453
23 23.393503189086914
24 21.825809478759766
25 25.668663024902344
26 25.250648498535156
27 26.36993408203125
28 22.522811889648438
29 22.357162475585938
30 25.221254348754883
31 23.928672790527344
32 24.199756622314453
33 27.24361228942871
34 22.592266082763672
35 23.011728286743164
36 18.40641975402832
37 26.18031883239746
38 24.040937423706055
39 26.852954864501953
40 25.80178451538086
41 24.644798278808594
42 23.33027458190918
43 22.914358139038086
44 25.911724090576172
45 22.30405044555664
46 22.05441856

In [29]:
g = torch.Generator().manual_seed(2147283647)

for _ in range(20):
    out = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))

crigiani.
harlemienni.
shiannteya.
kore.
nifmyra.
alandelia.
sha.
ityen.
jah.
delius.
isahiman.
medirinia.
isa.
maxielcel.
shia.
rosh.
ulan.
jazh.
brysee.
lynn.
